In [10]:
import os
import numpy as np
import rasterio
import matplotlib.pyplot as plt

from skimage.morphology import opening, closing, disk
from skimage.measure import label
from skimage.filters import threshold_otsu

import warnings
warnings.filterwarnings("ignore")

TRY1_DIR = "try-1"   # post-fire
TRY2_DIR = "try-2"   # pre-fire

# --- match event folders existing in both directories ---
events1 = set(os.listdir(TRY1_DIR))
events2 = set(os.listdir(TRY2_DIR))

events = sorted([
    e for e in events1.intersection(events2)
    if os.path.isdir(os.path.join(TRY1_DIR, e))
])

print("Matched events:", len(events))
if len(events) > 0:
    print("Example event:", events[0])
else:
    print("No matched events found!")

# --- find SentinelHub GeoTIFF one level below event folder ---
def find_tiff(event_folder):
    """
    Finds the first .tif/.tiff file exactly one level below the event folder.
    Assumes SentinelHub structure: event_xxx/<hash>/response.tiff
    """
    for sub in os.listdir(event_folder):
        sub_path = os.path.join(event_folder, sub)
        if not os.path.isdir(sub_path):
            continue

        for f in os.listdir(sub_path):
            f_low = f.lower()
            if f_low.endswith((".tif", ".tiff")) and "response" in f_low:
                return os.path.join(sub_path, f)

    return None

Matched events: 89
Example event: event_000


In [11]:
def load_sentinel_tiff(path):
    with rasterio.open(path) as src:
        img = src.read().astype(np.float32)
        meta = src.meta
    return img, meta


# Band indices (based on evalscript)
B02, B03, B04, B08, B12, SCL = 0, 1, 2, 3, 4, 5


def scl_valid_mask(scl):
    # Mask out: cloud shadow (3), water (6), cloud (8,9,10)
    invalid = np.isin(scl, [3, 6, 8, 9, 10])
    return ~invalid


def compute_nbr(nir, swir):
    eps = 1e-6
    return (nir - swir) / (nir + swir + eps)


def compute_dnbr(pre_img, post_img, valid_mask):
    nbr_pre  = compute_nbr(pre_img[B08],  pre_img[B12])
    nbr_post = compute_nbr(post_img[B08], post_img[B12])
    dnbr = nbr_pre - nbr_post
    dnbr[~valid_mask] = 0
    return dnbr


def dnbr_to_mask(dnbr):
    # Robust thresholding
    vals = dnbr[dnbr > 0]
    if vals.size == 0:
        return np.zeros_like(dnbr, dtype=bool)

    t = threshold_otsu(vals)
    mask = dnbr > t

    # Morphological cleanup
    mask = opening(mask, disk(2))
    mask = closing(mask, disk(3))

    return mask


def sam_stub_mask(post_img):
    # Light, conservative proxy (still optional)
    nir = post_img[B08]
    t = np.nanpercentile(nir, 65)
    return nir < t


def generate_gt_mask(dnbr_mask, sam_mask):
    return dnbr_mask & sam_mask


# -------- MANUAL DECISION VIEW --------
def show_manual_view(pre_img, post_img, valid_mask, dnbr, dnbr_mask, sam_mask, gt_mask):

    def norm01(x):
        x = x.astype(np.float32)
        x = x - np.nanmin(x)
        return x / (np.nanmax(x) + 1e-6)

    pre_rgb  = norm01(np.stack([pre_img[B04], pre_img[B03], pre_img[B02]], axis=-1))
    post_rgb = norm01(np.stack([post_img[B04], post_img[B03], post_img[B02]], axis=-1))
    false_color = norm01(np.stack([post_img[B08], post_img[B04], post_img[B03]], axis=-1))

    fig, axs = plt.subplots(2, 4, figsize=(18, 9))
    axs = axs.ravel()

    axs[0].imshow(pre_rgb);  axs[0].set_title("Pre RGB")
    axs[1].imshow(post_rgb); axs[1].set_title("Post RGB")

    im = axs[2].imshow(dnbr, cmap="inferno")
    axs[2].set_title("dNBR heatmap")
    plt.colorbar(im, ax=axs[2], fraction=0.046, pad=0.04)

    axs[3].imshow(false_color)
    axs[3].set_title("Post False Color (NIR,R,G)")

    axs[4].imshow(dnbr_mask, cmap="gray")
    axs[4].set_title("dNBR mask")

    axs[5].imshow(sam_mask, cmap="gray")
    axs[5].set_title("SAM mask")

    axs[6].imshow(post_rgb)
    axs[6].imshow(gt_mask, cmap="Reds", alpha=0.5)
    axs[6].set_title("GT = dNBR ∩ SAM")

    axs[7].axis("off")

    for ax in axs:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [12]:
accepted = []
gt_masks = {}

for event in events:

    print("\nProcessing event:", event)

    post_path = find_tiff(os.path.join(TRY1_DIR, event))
    pre_path  = find_tiff(os.path.join(TRY2_DIR, event))

    print("post path:", post_path)
    print("pre path :", pre_path)

    if post_path is None or pre_path is None:
        print("Missing pre/post image, skipping.")
        continue

    post_img, meta = load_sentinel_tiff(post_path)
    pre_img, _     = load_sentinel_tiff(pre_path)

    valid_mask = scl_valid_mask(post_img[SCL])

    dnbr = compute_dnbr(pre_img, post_img, valid_mask)
    dnbr_mask = dnbr_to_mask(dnbr)

    sam_mask = sam_stub_mask(post_img)
    gt_mask = generate_gt_mask(dnbr_mask, sam_mask)

    print("Showing manual view for", event)

    # --- MANUAL INSPECTION VIEW ---
    show_manual_view(pre_img, post_img, valid_mask, dnbr, dnbr_mask, sam_mask, gt_mask)

    # --- MANUAL DECISION ---
    decision = input(f"{event} accept GT? (y/n): ").strip().lower()
    if decision == "y":
        accepted.append(event)
        gt_masks[event] = gt_mask
        print("Accepted.")
    else:
        print("Rejected.")


Processing event: event_000
post path: try-1/event_000/faa212da38b73bc7b43ee43827a45c1b/response.tiff
pre path : try-2/event_000/f87c11a2b5463bceb45ed0c3e1a32b7d/response.tiff


ValueError: operands could not be broadcast together with shapes (256,) (258,) (256,) 

In [ ]:
os.makedirs("gt_masks", exist_ok=True)

for event, mask in gt_masks.items():
    out_path = f"gt_masks/{event}_gt.tif"
    meta.update(count=1, dtype="uint8")

    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(mask.astype(np.uint8), 1)

print("Saved", len(gt_masks), "GT masks")

Saved 0 GT masks


In [ ]:
def compute_f1_iou(pred, gt):
    tp = np.logical_and(pred, gt).sum()
    fp = np.logical_and(pred, ~gt).sum()
    fn = np.logical_and(~pred, gt).sum()

    f1 = 2 * tp / (2 * tp + fp + fn + 1e-6)
    iou = tp / (tp + fp + fn + 1e-6)
    return f1, iou